In [1]:
import numpy as np

In [11]:
class LassoRegressionScratch:
    def __init__(self, iterations = 1000, l1_penalty = 1):
        self.iterations = iterations
        self.l1_penalty = l1_penalty

    def predict(self, X):
        return np.dot(X, self.w) + self.b

    def soft_threshold(self, rho, lamda):
        if rho > lamda :
            return rho - lamda
        elif rho < -lamda:
            return rho + lamda
        else:
            return 0

    def fit(self, X, y):
        self.m, self.n = X.shape
        self.w = np.zeros(self.n)
        self.b = 0

        for i in range(self.iterations):
            #update bias like in standard linear regression
            y_pred = self.predict(X)
            self.b += np.mean(y - y_pred)

            for j in range(self.n):
                #Gauss Seidel method to use the most recent piece of info
                y_pred_without_j = self.predict(X) - X[:, j]*self.w[j]
                rho_j = np.dot(X[:, j], (y - y_pred_without_j))/self.m

                #soft threshold
                self.w[j] = self.soft_threshold(rho_j, self.l1_penalty)





In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

df = pd.read_csv('./Preprocessing/Feature_Encoded_Data.csv')

In [4]:
df = df.drop('Id', axis = 1)
print(f"null columns: {df.columns[df.isnull().any()].to_list()}")
print(f"data shape: {df.shape}")

null columns: []
data shape: (1458, 176)


In [12]:
X = df.drop('SalePrice', axis=1)
y = df['SalePrice']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
model = LassoRegressionScratch(iterations=1000, l1_penalty=1)
model.fit(X_train_scaled, y_train)
y_pred_train = model.predict(X_train_scaled)
y_pred = model.predict(X_test_scaled)
print(f'mse = {mean_squared_error(y_test, y_pred)}')
print(f'r2_train_score = {r2_score(y_train, y_pred_train)}')
print(f'r2_score = {r2_score(y_test, y_pred)}')

mse = 750815626.7186443
r2_train_score = 0.9245530246657206
r2_score = 0.864074459508718


In [13]:
X_train_scaled.shape

(1166, 175)

In [14]:
X_scaled = scaler.fit_transform(X)
model.fit(X_scaled, y)

In [15]:
real_test = pd.read_csv('./Data/test.csv')

In [16]:
#gotta clean the test data the same way we cleaned the train data
def clean(df):
    #All the changes I made myself
    
    # Make a copy to avoid modifying original
    data = df.copy()
    
    # Drop unnecessary columns (based on EDA: redundant, low variance, or weak correlation)
    columns_to_drop = [
        'Utilities', 'Street', 'Condition2', 'RoofMatl', 
        'BsmtFinSF1', 'BsmtFinSF2', 'Heating', 'LowQualFinSF', 
        '3SsnPorch', 'ScreenPorch', 'PoolArea', 'PoolQC', 
        'MiscFeature', 'MiscVal'
    ]
    data.drop(columns=columns_to_drop, axis=1, inplace=True, errors='ignore')
    
    # Transform KitchenAbvGr: group !=1 into 0 (1 kitchen vs multiple/none)
    data.loc[data['KitchenAbvGr'] != 1, 'KitchenAbvGr'] = 0
    
    # Transform GarageQual: group != 'TA' into 'Other'
    data.loc[data['GarageQual'] != 'TA', 'GarageQual'] = 'Other'
    
    # Transform GarageCond: group != 'TA' into 'Other'
    data.loc[data['GarageCond'] != 'TA', 'GarageCond'] = 'Other'
    
    # Transform EnclosedPorch: group >0 into 1 (has porch vs not)
    data.loc[data['EnclosedPorch'] > 0, 'EnclosedPorch'] = 1
    
    # Transform SaleType: group rare types into 'Other'
    data.loc[~data['SaleType'].isin(['WD', 'New', 'COD']), 'SaleType'] = 'Other'
    
    # Transform SaleCondition: group rare conditions into 'Other'
    data.loc[~data['SaleCondition'].isin(['Normal', 'Partial']), 'SaleCondition'] = 'Other'
    
    return data

def fillna(data):
    # Fill LotFrontage by neighborhood median
    data['LotFrontage'] = data.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))

    # Fill Alley with 'None'
    data["Alley"] = data["Alley"].fillna("None")

    # Fill MasVnrType and MasVnrArea
    data['MasVnrType'] = data['MasVnrType'].fillna('None')
    data['MasVnrArea'] = data['MasVnrArea'].fillna(0)

    # Fill basement columns
    data['BsmtQual'] = data['BsmtQual'].fillna('None')
    data['BsmtCond'] = data['BsmtCond'].fillna('None')
    data['BsmtExposure'] = data['BsmtExposure'].fillna('None')
    data['BsmtFinType1'] = data['BsmtFinType1'].fillna('None')
    data['BsmtFinType2'] = data['BsmtFinType2'].fillna('None')

    # Fill Electrical by neighborhood mode
    data['Electrical'] = data.groupby('Neighborhood')['Electrical'].transform(lambda x: x.fillna(x.mode()[0]))

    # Fill garage columns
    data['GarageType'] = data['GarageType'].fillna('None')
    data['GarageYrBlt'] = data['GarageYrBlt'].fillna(0)
    data['GarageFinish'] = data['GarageFinish'].fillna('None')
    data['GarageQual'] = data['GarageQual'].fillna('None')
    data['GarageCond'] = data['GarageCond'].fillna('None')

    # Fill other columns
    data['FireplaceQu'] = data['FireplaceQu'].fillna('None')
    data['Fence'] = data['Fence'].fillna('None')
    
    # Fill additional columns with nulls
    data['BsmtUnfSF'] = data['BsmtUnfSF'].fillna(0)
    data['TotalBsmtSF'] = data['TotalBsmtSF'].fillna(0)
    data['KitchenQual'] = data['KitchenQual'].fillna('TA')
    data['GarageCars'] = data['GarageCars'].fillna(0)
    data['GarageArea'] = data['GarageArea'].fillna(0)
    data['BsmtFullBath'] = data['BsmtFullBath'].fillna(0)
    data['BsmtHalfBath'] = data['BsmtHalfBath'].fillna(0)

    return data

def get_season(month):
    if month in [12,1,2]: 
        return 'Winter'
    elif month in [3,4,5]:
        return 'Spring'
    elif month in [6,7,8]:
        return 'Summer'
    else:
        return 'Fall'
    
def feature_engineering(df):
    #only keep one exterior column
    df['MixedExterior'] = (df['Exterior1st'] == df['Exterior2nd']).astype(int)
    df.drop('Exterior2nd', axis=1, inplace=True)
    #only keep total bath count
    df['TotalBath'] = (df['FullBath'] + df['BsmtFullBath']) + 0.5*(df['HalfBath'] + df['BsmtHalfBath'])
    df.drop('FullBath', axis=1, inplace=True)
    df.drop('BsmtFullBath', axis=1, inplace=True)
    df.drop('HalfBath', axis=1, inplace=True)
    df.drop('BsmtHalfBath', axis=1, inplace=True)
    #house age
    df['HouseAge'] = df['YrSold'] - df['YearBuilt']
    #year since remodeled
    df['YrSinceRemod'] = df['YrSold'] - df['YearRemodAdd']
    #whether the house was remodeled
    df['Remodeled'] = (df['YearBuilt'] == df['YearRemodAdd'])
    #delete year built and remodeled
    df.drop(['YearBuilt', 'YearRemodAdd'], axis=1, inplace=True)
    #garage condition doesn't matter much, only keep the quality
    df.drop('GarageCond', axis=1, inplace=True)
    #season sold
    df['SeasonSold'] = df['MoSold'].apply(get_season)
    df.drop('MoSold', axis=1, inplace=True)
    #total square feet
    df['TotalSF'] = df['1stFlrSF'] + df['2ndFlrSF'] + df['TotalBsmtSF']
    #new house
    df['NewHouse'] = (df['HouseAge'] < 2).astype(int)
    return df

def feature_encoding(df):
    categorical_cols = df.select_dtypes(include='object').columns.tolist()
    qual_map = {'None': 0, 'Other': 1, 'Po': 1, 'Fa':2, 'TA': 3, 'Gd': 4, 'Ex': 5}
    ordinal_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond','HeatingQC', 'KitchenQual', 'FireplaceQu', 'GarageQual']
    for col in ordinal_cols:
        df[col] = df[col].map(qual_map)
    nominal_cols = [col for col in categorical_cols if col not in ordinal_cols]
    df = pd.get_dummies(df, columns=nominal_cols, drop_first=True)
    return df

real_test = clean(real_test)
real_test = fillna(real_test)
real_test = feature_engineering(real_test)
real_test = feature_encoding(real_test)

In [17]:
print(f'null columns: {real_test.columns[real_test.isnull().any()]}')
print(f'test shape: {real_test.shape}')

null columns: Index([], dtype='object')
test shape: (1459, 172)


In [18]:
Id = real_test['Id']
real_test = real_test.drop('Id', axis = 1)
real_test = real_test.reindex(columns = X_train.columns, fill_value=0)

In [19]:
real_test_scaled = scaler.transform(real_test)

In [20]:
print(f'real_test_scaled shape: {real_test_scaled.shape}')
print(f'X_train_scaled shape: {X_train_scaled.shape}')

real_test_scaled shape: (1459, 175)
X_train_scaled shape: (1166, 175)


In [21]:
y_real_test_pred = model.predict(real_test_scaled)
submission_ridge = pd.DataFrame({
    "Id": Id,
    "SalePrice": y_real_test_pred
})

In [22]:
submission_ridge.to_csv('lasso_submission.csv')

In [ ]:
#this got a RMSLE score of 0.16151 on kaggle